# Phase B — Copy-Up-Baseline (Stufe 0)

Die **naive, lernfreie Baseline**: aus dem Tiefband wird das HF nur per spektraler
Hochkopie ergänzt (kein Modell). Sie ist gleichzeitig das *Sicherheitsnetz* und der
*Vergleichsanker*, gegen den die gelernten Stufen (Regression → GAN) gemessen werden.

Importiert die getesteten Bausteine aus `bwe/` — nicht self-contained.

In [ ]:
import os
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from bwe import config as cfg
from bwe.data import splits as SP
from bwe.data.loaders import load_demo
from bwe.infer.reconstruct import baseline_from_fullband
from bwe.eval import metrics as M
from bwe.eval import plots as P

name, target = load_demo("train", index=1, seconds=6.0, offset=30.0)
pred, inp = baseline_from_fullband(target)          # Rekonstruktion + bandbegrenzter Input
print(f"Demo-Track: {name}  |  {len(target) / cfg.SR:.1f} s")

## Spektrogramm-Tripel

Bandbegrenzt (Input) → Copy-Up-Rekonstruktion → Original. Über der weißen Linie (8 kHz) sieht man, wie das leere HF durch die kopierten Patches gefüllt wird.

In [ ]:
fig = P.spectro_triple(inp, pred, target)
plt.show()

### Crossover-Zoom

Detail um den Cutoff (8 kHz): die „Naht", an der das kopierte HF ansetzt.

In [ ]:
fig = P.crossover_zoom(pred, target)
plt.show()

## Anhören

Vorher (bandbegrenzt, dumpf) → Copy-Up (HF zurück, aber „kopiert") → Original.

In [ ]:
print("Bandbegrenzt (Input):"); display(Audio(inp.numpy(), rate=cfg.SR))
print("Copy-Up-Rekonstruktion:"); display(Audio(pred.numpy(), rate=cfg.SR))
print("Original (Target):");     display(Audio(target, rate=cfg.SR))

## Metriken

**LSD-HF** (Log-Spectral Distance über die HF-Bins, dB — kleiner = besser) ist die
Hauptzahl. **SI-SDR** (dB — größer = besser) ist der Wellenform-Sanity-Check. Zum
Vergleich auch die Werte des bandbegrenzten Inputs (leeres HF).

In [ ]:
print(f"{'':22s}{'LSD-HF [dB]':>14s}{'SI-SDR [dB]':>14s}")
print(f"{'Bandbegrenzt (Input)':22s}{M.lsd_hf(inp, target):14.2f}{M.si_sdr(inp, target):14.2f}")
print(f"{'Copy-Up':22s}{M.lsd_hf(pred, target):14.2f}{M.si_sdr(pred, target):14.2f}")

### Referenzzahl über die Val-Tracks

Die eingefrorene Baseline-Zahl, gegen die später Regression und GAN antreten (erste 60 s je Track).

In [ ]:
val_tracks = SP.get_split("valid")
print(f"{'Track':38s}{'LSD-HF in':>11s}{'LSD-HF CU':>11s}{'SI-SDR in':>11s}{'SI-SDR CU':>11s}")
acc = []
for i in range(len(val_tracks)):
    name, tgt = load_demo("valid", i, seconds=60.0, offset=0.0)
    pr, ip = baseline_from_fullband(tgt)
    row = (M.lsd_hf(ip, tgt), M.lsd_hf(pr, tgt), M.si_sdr(ip, tgt), M.si_sdr(pr, tgt))
    acc.append(row)
    print(f"{name[:36]:38s}{row[0]:11.2f}{row[1]:11.2f}{row[2]:11.2f}{row[3]:11.2f}")
m = np.mean(acc, axis=0)
print("-" * 82)
print(f"{'Mittel':38s}{m[0]:11.2f}{m[1]:11.2f}{m[2]:11.2f}{m[3]:11.2f}")

## Fazit

Copy-Up bringt das HF hörbar und sichtbar zurück und verbessert die **LSD-HF**
drastisch (≈100 → ≈16 dB) gegenüber dem leeren Input-HF. Es bleibt aber eine grobe
Hochkopie (wiederholte Patches, Crossover-Naht) — der **naive Anker**, den die
Regression (Stufe 1) und das GAN (Stufe 2) schlagen müssen.

**Schon hier zeigt sich die zentrale Pointe:** Die **SI-SDR wird durch Copy-Up
schlechter**, nicht besser — die Wellenform-Metrik bestraft das hinzugefügte HF
(falsche Phase/Inhalt), obwohl es voller *klingt*. Metrik und Ohr sind sich also
bereits an der Baseline uneinig; mit dem GAN wird diese Divergenz noch deutlicher.

> Hinweis: lokal nur wenige Val-Tracks (Subset). Die belastbare Zahl entsteht auf
> dem vollen Datensatz (Kaggle).